# 🏛️ MH-CET Merit List PDF → CSV Pipeline

Converts all CAP Round PDFs (CAP-I through CAP-IV) into structured CSVs, **mirroring the source folder structure**.

**Output columns per row:**
`cap_round | institute_code | institute_name | branch_code | branch_name | seat_pool | is_ews | is_tfws | institute_status | home_university | sanction_intake | cap_seats | ms_seats | minority_seats | ai_seats | sr_no | merit_no | mhtcet_score | application_id | candidate_name | gender | candidate_category | seat_type | is_vacant`

---
**Works on:** Local (macOS/Linux) | Google Colab | Kaggle Notebooks

## ⚙️ Cell 1 — Environment Detection & Package Install

In [ ]:
import sys, os, subprocess, importlib

# ── Detect environment ──────────────────────────────────────────────────────
IN_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle')
IS_LOCAL  = not IN_COLAB and not IN_KAGGLE

env_name = 'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')
print(f'🌍 Environment detected: {env_name}')

# ── Install Python packages ─────────────────────────────────────────────────
REQUIRED = ['pdfplumber', 'pandas', 'tqdm']

for pkg in REQUIRED:
    if importlib.util.find_spec(pkg) is None:
        print(f'📦 Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else:
        print(f'✅ {pkg} already available')

# ── Check pdftotext (system tool — used as fallback) ────────────────────────
def check_pdftotext():
    try:
        result = subprocess.run(['which', 'pdftotext'], capture_output=True, text=True)
        if result.returncode == 0:
            print('✅ pdftotext (poppler) found — will use as fallback')
            return True
    except Exception:
        pass
    # Try to install poppler on Colab/Linux
    if IN_COLAB or IN_KAGGLE:
        print('⬇️  Installing poppler-utils...')
        subprocess.run(['apt-get', 'install', '-y', '-q', 'poppler-utils'], capture_output=True)
        return True
    print('⚠️  pdftotext not found. Only pdfplumber will be used.')
    return False

HAS_PDFTOTEXT = check_pdftotext()
print('\n✅ Environment ready!')

## 🧱 Cell 1b — Databricks-Specific Setup (Skip if not on Databricks)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DATABRICKS SETUP CELL
# Run this cell ONLY when executing on a Databricks cluster.
# Skip it entirely on Local / Colab / Kaggle.
# ═══════════════════════════════════════════════════════════════════════

import sys
IN_DATABRICKS = 'dbruntime' in sys.modules or 'databricks' in str(type(spark)) if 'spark' in dir() else False

if not IN_DATABRICKS:
    print('⏭️  Not on Databricks — skip this cell')
else:
    # 1. Install Python libraries (cluster-scoped for this session)
    %pip install pdfplumber tqdm -q

    # 2. Install poppler (pdftotext fallback) via shell on each worker
    #    Note: For persistent installs across restarts, add this to your
    #    cluster Init Script instead (see SETUP.md → Databricks section)
    import subprocess
    subprocess.run(['sudo', 'apt-get', 'install', '-y', '-q', 'poppler-utils'], check=False)
    print('✅ poppler-utils installed on driver')

    # 3. Mount cloud storage (edit ONE of the blocks below for your cloud)
    # ── Option A: Azure Data Lake Storage Gen2 (ADLS) ──────────────────
    # configs = {
    #     'fs.azure.account.auth.type': 'OAuth',
    #     'fs.azure.account.oauth.provider.type': 'org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider',
    #     'fs.azure.account.oauth2.client.id': '<your-client-id>',
    #     'fs.azure.account.oauth2.client.secret': dbutils.secrets.get(scope='kv', key='sp-secret'),
    #     'fs.azure.account.oauth2.client.endpoint': 'https://login.microsoftonline.com/<tenant-id>/oauth2/token',
    # }
    # dbutils.fs.mount(
    #     source='abfss://<container>@<storage-account>.dfs.core.windows.net/',
    #     mount_point='/mnt/mahacet',
    #     extra_configs=configs
    # )

    # ── Option B: AWS S3 ────────────────────────────────────────────────
    # dbutils.fs.mount(
    #     source='s3a://<your-bucket>/mahacet/',
    #     mount_point='/mnt/mahacet',
    #     extra_configs={
    #         'fs.s3a.access.key': dbutils.secrets.get(scope='aws', key='access-key'),
    #         'fs.s3a.secret.key': dbutils.secrets.get(scope='aws', key='secret-key'),
    #     }
    # )

    # ── Option C: GCS ───────────────────────────────────────────────────
    # dbutils.fs.mount(
    #     source='gs://<your-bucket>/mahacet/',
    #     mount_point='/mnt/mahacet',
    # )

    # 4. Verify mount
    # display(dbutils.fs.ls('/mnt/mahacet/mahacet_cutoffs_2025/'))

    # 5. Override environment flag so Cell 2 picks Databricks paths
    IN_COLAB  = False
    IN_KAGGLE = False
    IS_LOCAL  = False
    IN_DATABRICKS = True
    print('✅ Databricks environment ready')
    print('👉 Next: verify your mount, then update PDF_ROOT in Cell 2')


## 📁 Cell 2 — Configuration (Edit These Paths)

In [ ]:
from pathlib import Path
IN_DATABRICKS = True
IN_COLAB = False
IN_KAGGLE = False
IS_LOCAL = False

PDF_ROOT = Path("/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025")
CSV_ROOT = PDF_ROOT.parent / "mahacet_cutoffs_2025_csv"
LOG_DIR  = PDF_ROOT.parent / "logs"
SKIP_EXISTING  = True
VALIDATE_ROWS  = True
INCLUDE_VACANT = True

assert PDF_ROOT.exists(), f"PDF root not found: {PDF_ROOT}"
rounds = sorted([d.name for d in PDF_ROOT.iterdir() if d.is_dir() and d.name.startswith("CAP")])
total_pdfs = sum(len(list((PDF_ROOT/r).glob("*.pdf"))) for r in rounds)
print(f"PDF Root  : {PDF_ROOT}")
print(f"CSV Root  : {CSV_ROOT}")
print(f"Log Dir   : {LOG_DIR}")
print(f"Rounds    : {rounds}")
print(f"Total PDFs: {total_pdfs}")


## 🔍 Cell 3 — PDF Parser (Core Engine)

In [ ]:
import re
import threading
import pdfplumber
import subprocess
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

# ── Constants ────────────────────────────────────────────────────────────────
KNOWN_SEAT_TYPES = {
    'GOPENS','LOPENS','GOBCS','LOBCS','GSCS','LSCS','GSTS','LSTS',
    'GNT1S','LNT1S','GNT2S','LNT2S','GNT3S','LNT3S','GVJS','LVJS',
    'GSEBCS','LSEBCS','GEWSS','LEWSS',
    'PWDROPENS','PWDROBCS','PWDRSCS','PWDRSTS','PWDRSEBCS',
    'DEFROPENS','DEFROBCS','DEFRSCS','DEFRSTS','DEFRSEBCS',
    'DEFRNT3S','DEFRNT2S','DEFRNT1S',
    'DEFOBCS','DEFOPENS','PWDOBCS','PWDOPENS',
    'DEFSCS','DEFSEBCS','PWDSCS','PWDSEBCS',
    'DEFNT1S','DEFNT2S','DEFNT3S','DEFVJS',
    'PWDNT1S','PWDNT2S','PWDNT3S','PWDVJS',
    'ORPHAN','TFWS','EWS','AI','MI',
}

SEAT_MARKERS = {'^', '~', '*', '@', '&'}

RE_APP_ID    = re.compile(r'EN\d{8}')
RE_SCORE     = re.compile(r'\d{2,6}\.\d+')
RE_MERIT_NO  = re.compile(r'\b(\d{1,6})\b')
RE_INSTITUTE = re.compile(r'^(\d{5})\s+(.+)$')
RE_BRANCH    = re.compile(r'^(\d{10}(?:\[EWS\])?T?)\s*-\s*(.+)$')
RE_STATUS    = re.compile(r'^Status:\s*(.+?)(?:\s+Home University\s*:\s*(.+))?$')
RE_INTAKE    = re.compile(r'Sanction Intake:\s*(\d+)')
RE_CAP       = re.compile(r'CAP Seats:\s*(\d+)')
RE_MS        = re.compile(r'MS Seats:\s*(\d+)')
RE_MIN       = re.compile(r'Minority Seats\s*:\s*(\d+)')
RE_AI        = re.compile(r'AI Seats:\s*(\d+)')
RE_GENDER    = re.compile(r'\b([MF])\b')

SKIP_PATTERNS = [
    re.compile(r'^Government of Maharashtra'),
    re.compile(r'^State Common Entrance Test Cell'),
    re.compile(r'Provisional Allotment List'),
    re.compile(r'Degree Courses'),
    re.compile(r'Integrated 5 Years'),
    re.compile(r'^(P\s+)?Meri\s+Sr\.'),
    re.compile(r'^No\.\s+No\.'),
    re.compile(r'^State Level Seats'),
    re.compile(r'^All India Seats'),
    re.compile(r'Legends for SeatType'),
    re.compile(r'Legends for ChoiceCode'),
    re.compile(r'^Seat Type\s*:'),
    re.compile(r'Red Color|Black Color|Green Color|Blue Color|Gray Color'),
    re.compile(r'^MI-Minority'),
    re.compile(r'^PWDR\s*:'),
    re.compile(r'^DEFR\s*:'),
    re.compile(r'^Merit No\s*:'),
    re.compile(r'^Page \d+ of \d+'),
    re.compile(r'^Master of Engineering'),
    re.compile(r'for the Year 20'),
    re.compile(r'for the Admission'),
]

def should_skip(line):
    return any(p.search(line) for p in SKIP_PATTERNS)

def strip_marker(token):
    if token and token[0] in SEAT_MARKERS:
        return token[1:].strip(), token[0]
    return token, ''

def find_seat_type(tokens):
    for i, t in enumerate(tokens):
        clean, marker = strip_marker(t)
        if clean in KNOWN_SEAT_TYPES:
            return clean, marker, i
    return '', '', -1

def parse_row_line(line, institute_code, institute_name, branch):
    """
    Parse a single full row line like:
    '1 5369 98.4177884 EN25154658 RITHE RIDDHIMA SANTOSH F OBC LOPENS'
    or with marker:
    '1 5369 98.4177884 EN25154658 RITHE RIDDHIMA SANTOSH F OBC ^ LOPENS'
    or VACANT:
    '27 VACANT MI'
    """
    line = line.strip()
    if not line:
        return None

    # VACANT row
    if 'VACANT' in line:
        tokens = line.split()
        sr_no = tokens[0] if tokens and tokens[0].isdigit() else ''
        seat_type = next((t for t in tokens if t in KNOWN_SEAT_TYPES), '')
        seat_marker = next((t for t in tokens if t in SEAT_MARKERS), '')
        return _make_row(institute_code, institute_name, branch,
                        sr_no=sr_no, merit_no='', score='', app_id='VACANT',
                        name='VACANT', gender='', category='',
                        seat_type=seat_type, seat_marker=seat_marker, is_vacant=True)

    # Must contain an application ID
    app_match = RE_APP_ID.search(line)
    if not app_match:
        return None

    app_id  = app_match.group()
    app_pos = app_match.start()
    before  = line[:app_pos].strip()
    after   = line[app_pos + len(app_id):].strip()

    # ── Parse BEFORE (sr_no, merit_no, score) ────────────────────────────────
    nums_before = re.findall(r'\d+(?:\.\d+)?', before)
    score   = next((n for n in nums_before if '.' in n), '')
    int_nums = [n for n in nums_before if '.' not in n]
    sr_no   = int_nums[0] if len(int_nums) >= 1 else ''
    merit_no = int_nums[1] if len(int_nums) >= 2 else ''

    # ── Parse AFTER (name, gender, category, seat_type) ──────────────────────
    # seat_type is always last (possibly with marker prefix)
    tokens_after = after.split()

    seat_type   = ''
    seat_marker = ''
    gender      = ''
    category    = ''
    name        = ''

    # Find seat type from the end
    for j in range(len(tokens_after) - 1, -1, -1):
        clean, marker = strip_marker(tokens_after[j])
        if clean in KNOWN_SEAT_TYPES:
            seat_type   = clean
            seat_marker = marker
            tokens_after = tokens_after[:j]
            # also remove a lone marker token just before seat type
            if tokens_after and tokens_after[-1] in SEAT_MARKERS:
                seat_marker = tokens_after[-1]
                tokens_after = tokens_after[:-1]
            break

    # Find gender (M or F as standalone token)
    for j in range(len(tokens_after) - 1, -1, -1):
        t = tokens_after[j]
        if t in ('M', 'F'):
            gender = t
            category_tokens = tokens_after[j+1:]
            name_tokens     = tokens_after[:j]
            category = ' '.join(category_tokens).strip()
            name     = ' '.join(name_tokens).strip()
            break

    if not gender:
        name = ' '.join(tokens_after).strip()

    # Validate
    warnings = []
    if not app_id or not RE_APP_ID.match(app_id): warnings.append(f'bad_app_id:{app_id}')
    if not score:   warnings.append('missing_score')
    if not gender:  warnings.append('missing_gender')
    if not seat_type: warnings.append('missing_seat_type')
    if not name:    warnings.append('missing_name')

    row = _make_row(institute_code, institute_name, branch,
                    sr_no=sr_no, merit_no=merit_no, score=score,
                    app_id=app_id, name=name, gender=gender,
                    category=category, seat_type=seat_type,
                    seat_marker=seat_marker, is_vacant=False)
    row['_validation_warnings'] = '|'.join(warnings)
    return row

def _make_row(inst_code, inst_name, branch, **kwargs):
    return {
        'cap_round':          branch.get('cap_round',''),
        'institute_code':     inst_code,
        'institute_name':     inst_name,
        'branch_code':        branch.get('branch_code',''),
        'branch_name':        branch.get('branch_name',''),
        'is_ews':             branch.get('is_ews', False),
        'is_tfws':            branch.get('is_tfws', False),
        'institute_status':   branch.get('institute_status',''),
        'home_university':    branch.get('home_university',''),
        'sanction_intake':    branch.get('sanction_intake',''),
        'cap_seats':          branch.get('cap_seats',''),
        'ms_seats':           branch.get('ms_seats',''),
        'minority_seats':     branch.get('minority_seats',''),
        'ai_seats':           branch.get('ai_seats',''),
        'sr_no':              kwargs.get('sr_no',''),
        'merit_no':           kwargs.get('merit_no',''),
        'mhtcet_score':       kwargs.get('score',''),
        'application_id':     kwargs.get('app_id',''),
        'candidate_name':     kwargs.get('name',''),
        'gender':             kwargs.get('gender',''),
        'candidate_category': kwargs.get('category',''),
        'seat_type':          kwargs.get('seat_type',''),
        'seat_marker':        kwargs.get('seat_marker',''),
        'is_vacant':          kwargs.get('is_vacant', False),
        '_validation_warnings': '',
    }

def extract_text_pdfplumber(pdf_path):
    pages = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=3, y_tolerance=3)
            if text:
                pages.append(text)
    return '\n'.join(pages)

def parse_pdf(pdf_path, cap_round):
    try:
        raw = extract_text_pdfplumber(pdf_path)
    except Exception as e:
        return [], [f'extraction_failed: {e}'], 'failed'

    if not raw.strip():
        return [], ['empty_text'], 'pdfplumber'

    lines = raw.split('\n')
    rows = []
    warnings = []
    institute_code = ''
    institute_name = ''
    branch = {'cap_round': cap_round}

    for line in lines:
        line = line.strip()
        if not line or should_skip(line):
            continue

        # Institute header
        m = RE_INSTITUTE.match(line)
        if m and len(m.group(1)) == 5 and not RE_APP_ID.search(line):
            institute_code = m.group(1)
            institute_name = m.group(2).strip()
            continue

        # Branch header
        m = RE_BRANCH.match(line)
        if m:
            raw_code  = m.group(1)
            is_ews    = '[EWS]' in raw_code
            is_tfws   = raw_code.endswith('T') and not is_ews
            base_code = re.sub(r'\[EWS\]|T$', '', raw_code)
            branch = {
                'cap_round':        cap_round,
                'branch_code':      base_code,
                'branch_name':      m.group(2).strip(),
                'is_ews':           is_ews,
                'is_tfws':          is_tfws,
                'institute_status': branch.get('institute_status',''),
                'home_university':  branch.get('home_university',''),
                'sanction_intake':  branch.get('sanction_intake',''),
                'cap_seats':        branch.get('cap_seats',''),
                'ms_seats':         branch.get('ms_seats',''),
                'minority_seats':   branch.get('minority_seats',''),
                'ai_seats':         branch.get('ai_seats',''),
            }
            continue

        # Status
        m = RE_STATUS.match(line)
        if m:
            branch['institute_status'] = (m.group(1) or '').strip()
            branch['home_university']  = (m.group(2) or '').strip()
            continue

        # Seat metadata line (all on one line on Databricks)
        if RE_INTAKE.search(line):  branch['sanction_intake'] = RE_INTAKE.search(line).group(1)
        if RE_CAP.search(line):     branch['cap_seats']       = RE_CAP.search(line).group(1)
        if RE_MS.search(line):      branch['ms_seats']        = RE_MS.search(line).group(1)
        if RE_MIN.search(line):     branch['minority_seats']  = RE_MIN.search(line).group(1)
        if RE_AI.search(line):      branch['ai_seats']        = RE_AI.search(line).group(1)

        if any(p.search(line) for p in [RE_INTAKE, RE_CAP, RE_MS, RE_MIN, RE_AI]):
            continue

        # Candidate row — must contain an app ID or VACANT
        if not RE_APP_ID.search(line) and 'VACANT' not in line:
            continue

        row = parse_row_line(line, institute_code, institute_name, branch)
        if row:
            if row.get('_validation_warnings'):
                warnings.append(f"{row['application_id']}: {row['_validation_warnings']}")
            rows.append(row)

    return rows, warnings, 'pdfplumber'

print('✅ New parser loaded')


## 🧪 Cell 4 — Smoke Test on a Single PDF

In [ ]:
import pandas as pd

test_cases = [
    (PDF_ROOT / 'CAP-I'  / 'CAPR-I_01002.pdf',  'CAP-I'),
    (PDF_ROOT / 'CAP-I'  / 'CAPR-I_01120.pdf',  'CAP-I'),   # minority + VACANT
    (PDF_ROOT / 'CAP-II' / 'CAPR-II_01002.pdf', 'CAP-II'),  # markers
    (PDF_ROOT / 'CAP-IV' / 'CAPR-IV_01002.pdf', 'CAP-IV'),
]

for pdf_path, cap_round in test_cases:
    rows, warnings, method = parse_pdf(pdf_path, cap_round)
    df = pd.DataFrame(rows) if rows else pd.DataFrame()
    vacants  = len(df[df['is_vacant']==True]) if len(df) else 0
    no_seat  = len(df[(df['seat_type']=='') & (df['is_vacant']==False)]) if len(df) else 0
    markers  = set(df['seat_marker'].unique()) - {''} if len(df) else set()
    status   = '✅' if len(df) > 0 and no_seat < 10 else '❌'
    print(f"{status} {pdf_path.name}: {len(df)} rows | {vacants} vacant | {no_seat} no-seat | markers={markers}")
    if len(df):
        r = df.iloc[0]
        print(f"   → {r['application_id']} | {r['candidate_name']!r} | {r['gender']} | {r['candidate_category']} | {r['seat_type']}")


## 🚀 Cell 5 — Full Batch Conversion Pipeline

In [ ]:
import json, time, threading, pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

CSV_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

run_ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = LOG_DIR / f'conversion_{run_ts}.jsonl'
fail_log = LOG_DIR / f'failures_{run_ts}.txt'

# Collect all PDFs
all_pdfs = []
for round_dir in sorted(PDF_ROOT.iterdir()):
    if not round_dir.is_dir() or not round_dir.name.startswith('CAP'):
        continue
    for pdf in sorted(round_dir.glob('*.pdf')):
        csv_out = CSV_ROOT / round_dir.name / (pdf.stem + '.csv')
        all_pdfs.append((pdf, round_dir.name, csv_out))

# Resume mode
pending = [(p, r, c) for p, r, c in all_pdfs if not c.exists()]
print(f'⏭️  Skipping {len(all_pdfs) - len(pending)} already done')
print(f'🔄 Converting {len(pending)} / {len(all_pdfs)} PDFs with 2 workers')

_log_lock = threading.Lock()
stats = {'success': 0, 'failed': 0, 'total_rows': 0,
         'skipped': len(all_pdfs) - len(pending), 'total': len(all_pdfs), 'failures': []}

def convert_one(args):
    pdf_path, cap_round, csv_out = args
    t0 = time.time()
    try:
        rows, warnings, method = parse_pdf(pdf_path, cap_round)
        if not rows:
            raise ValueError('Parser returned 0 rows')
        df = pd.DataFrame(rows)
        for col in ['sr_no','merit_no','sanction_intake','cap_seats','ms_seats','minority_seats','ai_seats']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['mhtcet_score'] = pd.to_numeric(df['mhtcet_score'], errors='coerce')
        df['is_ews']    = df['is_ews'].astype(bool)
        df['is_tfws']   = df['is_tfws'].astype(bool)
        df['is_vacant'] = df['is_vacant'].astype(bool)
        csv_out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_out, index=False, encoding='utf-8-sig')
        return {'status':'ok','file':pdf_path.name,'round':cap_round,
                'rows':len(df),'warnings':len(warnings),'method':method,
                'elapsed_s':round(time.time()-t0,2),'ts':datetime.now().isoformat()}
    except Exception as e:
        return {'status':'failed','file':pdf_path.name,'round':cap_round,
                'rows':0,'error':f'{type(e).__name__}: {e}',
                'elapsed_s':round(time.time()-t0,2),'ts':datetime.now().isoformat()}

with open(log_file,'w') as lf, open(fail_log,'w') as ff:
    ff.write(f'Failures — {run_ts}\n{"-"*60}\n')
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(convert_one, args): args for args in pending}
        done_count = 0
        for future in as_completed(futures):
            result = future.result()
            done_count += 1
            with _log_lock:
                lf.write(json.dumps(result) + '\n')
                if result['status'] == 'ok':
                    stats['success']    += 1
                    stats['total_rows'] += result['rows']
                else:
                    stats['failed'] += 1
                    stats['failures'].append(result)
                    ff.write(f"{result['file']} | {result['round']} | {result.get('error','')}\n")
            if done_count % 50 == 0 or done_count == len(pending):
                print(f"  Progress: {done_count}/{len(pending)} | ✅ {stats['success']} | ❌ {stats['failed']}")

print(f"\n{'='*60}")
print(f"✅ DONE")
print(f"  Total PDFs    : {stats['total']}")
print(f"  Skipped       : {stats['skipped']}")
print(f"  Succeeded     : {stats['success']}")
print(f"  Failed        : {stats['failed']}")
print(f"  Total rows    : {stats['total_rows']:,}")
print(f"  Log           : {log_file}")


## 📊 Cell 6 — Post-Conversion Report & Validation Summary

In [ ]:
# ── Read all log entries ──────────────────────────────────────────────────────
import glob
import pandas as pd
import json

log_entries = []
for lf in sorted(glob.glob(str(LOG_DIR / 'conversion_*.jsonl'))):
    with open(lf) as f:
        for line in f:
            line = line.strip()
            if line:
                log_entries.append(json.loads(line))

if not log_entries:
    print('No log entries found — run Cell 5 first.')
else:
    df_log = pd.DataFrame(log_entries)

    print('📊 CONVERSION REPORT')
    print('='*60)

    # ── Per round summary ─────────────────────────────────────────────────────
    summary = df_log.groupby(['round', 'status']).agg(
        files=('file', 'count'),
        total_rows=('rows', 'sum'),
        avg_rows=('rows', 'mean'),
        avg_time_s=('elapsed_s', 'mean'),
    ).round(2)
    display(summary)

    # ── Extraction method breakdown ───────────────────────────────────────────
    print('\n🔧 Extraction method used:')
    ok = df_log[df_log['status'] == 'ok']
    if 'method' in ok.columns:
        print(ok['method'].value_counts().to_string())

    # ── Files with validation warnings ───────────────────────────────────────
    print('\n⚠️  Files with validation warnings:')
    if 'warnings' in df_log.columns:
        warned = df_log[(df_log['status'] == 'ok') & (df_log['warnings'] > 0)]
        if len(warned):
            display(warned[['file', 'round', 'rows', 'warnings']]
                    .sort_values('warnings', ascending=False)
                    .head(20)
                    .reset_index(drop=True))
        else:
            print('  ✅ No warnings')
    else:
        print('  (no warnings column in log)')

    # ── Failed files ──────────────────────────────────────────────────────────
    failed = df_log[df_log['status'] == 'failed']
    if len(failed):
        print(f'\n❌ Failed files ({len(failed)}):')
        display(failed[['file', 'round', 'error']].reset_index(drop=True))
    else:
        print('\n🎉 No failures!')

    # ── Overall totals ────────────────────────────────────────────────────────
    print('\n' + '═'*60)
    print(f"  Total files processed : {len(df_log)}")
    print(f"  ✅ Succeeded          : {len(df_log[df_log['status']=='ok'])}")
    print(f"  ❌ Failed             : {len(failed)}")
    print(f"  📊 Total rows         : {int(df_log['rows'].sum()):,}")
    print(f"  ⏱️  Avg time per PDF   : {df_log['elapsed_s'].mean():.2f}s")
    print(f"  ⏱️  Total time         : {df_log['elapsed_s'].sum()/60:.1f} mins")


## 🗺️ Cell 7 — Verify Output Folder Structure

In [ ]:
print(f'📂 Output structure: {CSV_ROOT}\n')

total_csvs = 0
for round_dir in sorted(CSV_ROOT.iterdir()):
    if not round_dir.is_dir():
        continue
    csvs = list(round_dir.glob('*.csv'))
    total_csvs += len(csvs)
    pdfs_in_round = len(list((PDF_ROOT / round_dir.name).glob('*.pdf')))
    pct = round(len(csvs)/pdfs_in_round*100, 1) if pdfs_in_round else 0
    print(f'  {round_dir.name}/  →  {len(csvs)}/{pdfs_in_round} CSVs ({pct}%)')

print(f'\n  Total CSVs created: {total_csvs}')

# Quick peek at one CSV
sample_csvs = list(CSV_ROOT.glob('**/*.csv'))
if sample_csvs:
    print(f'\n📋 Sample CSV preview: {sample_csvs[0].name}')
    display(pd.read_csv(sample_csvs[0]).head(5))

## 🔄 Cell 8 — Self-Heal: Retry Failed PDFs

In [ ]:
# This cell re-runs ONLY previously failed PDFs.
# It reads the latest failure log and retries each one.

fail_logs = sorted(glob.glob(str(LOG_DIR / 'failures_*.txt')))
if not fail_logs:
    print('✅ No failure logs found — nothing to retry!')
else:
    latest_fail_log = fail_logs[-1]
    print(f'📄 Reading failures from: {latest_fail_log}')

    retry_queue = []
    with open(latest_fail_log) as f:
        for line in f:
            line = line.strip()
            if '|' not in line or line.startswith('Failure'):
                continue
            parts = line.split(' | ')
            if len(parts) >= 2:
                fname, cap_round = parts[0].strip(), parts[1].strip()
                pdf_path = PDF_ROOT / cap_round / fname
                csv_out  = CSV_ROOT / cap_round / (Path(fname).stem + '.csv')
                if pdf_path.exists():
                    retry_queue.append((pdf_path, cap_round, csv_out))

    print(f'🔁 Retrying {len(retry_queue)} failed PDF(s)...')
    retry_stats = {'success': 0, 'still_failed': 0, 'failures': []}

    for pdf_path, cap_round, csv_out in tqdm(retry_queue, desc='Retrying'):
        try:
            rows, warnings, method = parse_pdf(pdf_path, cap_round)
            if not rows:
                raise ValueError('0 rows after retry')
            df = pd.DataFrame(rows)
            csv_out.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(csv_out, index=False, encoding='utf-8-sig')
            retry_stats['success'] += 1
            print(f'  ✅ {pdf_path.name} → {len(df)} rows')
        except Exception as e:
            retry_stats['still_failed'] += 1
            reason = f'{type(e).__name__}: {e}'
            retry_stats['failures'].append({'file': pdf_path.name, 'reason': reason})
            print(f'  ❌ {pdf_path.name}: {reason}')

    print(f'\n🔁 Retry results: {retry_stats["success"]} recovered, {retry_stats["still_failed"]} still failing')
    if retry_stats['failures']:
        print('\nStill failing files (manual inspection needed):')
        for f in retry_stats['failures']:
            print(f"  {f['file']}: {f['reason']}")